# Лабораторная работа 11. PySpark

**Задание:**
1. Классификация цветков ирисов с использованием PySpark
2. Классификация пассажиров титаника с использованием PySpark

**Ограничения:** без pandas, без sklearn — только `pyspark.ml`

## Инициализация PySpark

PySpark требует установленную Java. Прописываем `JAVA_HOME` вручную, чтобы Python-процесс нашёл JVM до создания SparkSession.

In [ ]:
import os

JAVA_HOME = r"C:\Program Files\Java\jdk-21"
os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["PATH"] = os.path.join(JAVA_HOME, "bin") + os.pathsep + os.environ.get("PATH", "")
print("JAVA_HOME:", os.environ["JAVA_HOME"])

In [ ]:
from pyspark.sql import SparkSession

# local[*] — запускаем Spark локально, используя все доступные ядра процессора
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark

---
# Часть 1. Классификация цветков ирисов

**Датасет Iris** содержит 150 записей о трёх видах ирисов (Setosa, Versicolor, Virginica).  
Каждый цветок описан 4 числовыми признаками:

`sepal_length` / `sepal_width` — длина и ширина чашелистика <br/>
`petal_length` / `petal_width` — длина и ширина лепестка

**Задача:** по 4 признакам предсказать вид цветка (3 класса).

### Загрузка данных

In [ ]:
# Колонки в файле содержат точки (sepal.length) — PySpark ML их не принимает.
# toDF() переименовывает все колонки сразу в нужном порядке.
iris_df = spark.read.csv('iris.csv', inferSchema=True, header=True)
iris_df = iris_df.toDF('sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'variety')

iris_df.show(5)
iris_df.printSchema()
print('Количество записей по каждому классу:')
iris_df.groupBy('variety').count().show()

### Предобработка и обучение модели

PySpark ML требует два обязательных столбца:
- **`label`** — числовой номер класса (не строка)
- **`features`** — все признаки в виде одного вектора

Для этого используем три инструмента:

| Инструмент | Что делает |
|---|---|
| `StringIndexer` | Переводит строку `variety` в число: Setosa→0, Versicolor→1, Virginica→2 |
| `VectorAssembler` | Собирает 4 числовых столбца в один столбец-вектор `features` |
| `LogisticRegression` | Обучает классификатор на столбцах `features` и `label` |

**Pipeline** — это конвейер, который применяет все шаги последовательно: сначала индексирует метки, потом собирает вектор, потом обучает модель.

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

label_indexer = StringIndexer(inputCol='variety', outputCol='label')

assembler = VectorAssembler(
    inputCols=['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
    outputCol='features'
)

lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100)

iris_pipeline = Pipeline(stages=[label_indexer, assembler, lr])

# Делим данные: 80% на обучение, 20% на тест
iris_train, iris_test = iris_df.randomSplit([0.8, 0.2], seed=42)
iris_model = iris_pipeline.fit(iris_train)
print('Модель обучена')

### Предсказания и оценка качества

In [ ]:
# transform() применяет обученный pipeline к тестовым данным
iris_pred = iris_model.transform(iris_test)
iris_pred.select('variety', 'label', 'prediction').show()

In [ ]:
# MulticlassClassificationEvaluator считает точность для задач с 3+ классами
evaluator = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='accuracy'
)
accuracy = evaluator.evaluate(iris_pred)

total   = iris_pred.count()
correct = iris_pred.filter(F.col('label') == F.col('prediction')).count()

print(f'Iris — Accuracy: {accuracy * 100:.1f}%')
print(f'Правильно: {correct} из {total}')
print('\nМатрица ошибок (строка = реальный класс, столбец prediction = предсказанный):')
iris_pred.groupBy('variety', 'prediction').count().orderBy('variety').show()

---
# Часть 2. Классификация пассажиров Титаника

**Датасет Titanic** содержит данные о 891 пассажире.  
Каждый пассажир описан несколькими признаками:

| Признак | Описание |
|---|---|
| `Survived` | **Метка класса**: 1 = выжил, 0 = погиб |
| `Pclass` | Класс билета (1, 2, 3) |
| `Sex` | Пол (male / female) |
| `Age` | Возраст |
| `SibSp` | Кол-во братьев/сестёр и супругов на борту |
| `Parch` | Кол-во родителей и детей на борту |
| `Fare` | Стоимость билета |
| `Embarked` | Порт посадки (S / C / Q) |

**Задача:** предсказать выжил пассажир или нет (2 класса — бинарная классификация).

### Загрузка данных

In [ ]:
titanic_df = spark.read.csv('titanic.csv', inferSchema=True, header=True)
titanic_df.show(5)
titanic_df.printSchema()

### Предобработка данных

В данных есть пропуски (особенно в `Age` и `Embarked`) — удаляем такие строки через `dropna()`.  
Оставляем только те признаки, которые имеют смысл для предсказания выживания.

In [ ]:
titanic_clean = titanic_df \
    .select('Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked') \
    .dropna()

print('Строк после удаления пропусков:', titanic_clean.count())
print('Распределение по классу (0=погиб, 1=выжил):')
titanic_clean.groupBy('Survived').count().show()

### Построение Pipeline

В Titanic есть строковые категориальные признаки (`Sex`, `Embarked`), которые нужно закодировать:

| Шаг | Инструмент | Что делает |
|---|---|---|
| 1 | `StringIndexer` | `Sex`: male→0, female→1 / `Embarked`: S→0, C→1, Q→2 |
| 2 | `OneHotEncoder` | Превращает индексы в бинарные векторы — убирает ложную «упорядоченность» чисел |
| 3 | `VectorAssembler` | Собирает все признаки в один вектор `features` |
| 4 | `LogisticRegression` | Обучает бинарный классификатор |

> **Почему OneHotEncoder?** Если оставить `Sex` как 0/1, модель может решить что female «больше» male на единицу. OHE создаёт независимые бинарные признаки для каждого значения категории.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml import Pipeline

sex_indexer      = StringIndexer(inputCol='Sex',      outputCol='sex_idx')
embarked_indexer = StringIndexer(inputCol='Embarked', outputCol='embarked_idx')

sex_encoder      = OneHotEncoder(inputCol='sex_idx',      outputCol='sex_ohe')
embarked_encoder = OneHotEncoder(inputCol='embarked_idx', outputCol='embarked_ohe')

assembler = VectorAssembler(
    inputCols=['Pclass', 'sex_ohe', 'Age', 'SibSp', 'Parch', 'Fare', 'embarked_ohe'],
    outputCol='features'
)

lr = LogisticRegression(featuresCol='features', labelCol='Survived', maxIter=100)

titanic_pipeline = Pipeline(stages=[
    sex_indexer, embarked_indexer,
    sex_encoder, embarked_encoder,
    assembler, lr
])

titanic_train, titanic_test = titanic_clean.randomSplit([0.8, 0.2], seed=42)
titanic_model = titanic_pipeline.fit(titanic_train)
print('Модель обучена')

### Предсказания и оценка качества

In [ ]:
titanic_pred = titanic_model.transform(titanic_test)
titanic_pred.select('Survived', 'prediction', 'probability').show(10)

In [ ]:
# Accuracy — доля правильных ответов
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol='Survived', predictionCol='prediction', metricName='accuracy'
)
# AUC-ROC — насколько хорошо модель разделяет классы (1.0 = идеально, 0.5 = случайно)
auc_evaluator = BinaryClassificationEvaluator(
    labelCol='Survived', rawPredictionCol='rawPrediction', metricName='areaUnderROC'
)

accuracy = acc_evaluator.evaluate(titanic_pred)
auc      = auc_evaluator.evaluate(titanic_pred)

total   = titanic_pred.count()
correct = titanic_pred.filter(
    F.col('Survived') == F.col('prediction').cast('int')
).count()

print(f'Titanic — Accuracy: {accuracy * 100:.1f}%')
print(f'Titanic — AUC-ROC:  {auc:.4f}')
print(f'Правильно: {correct} из {total}')
print('\nМатрица ошибок:')
titanic_pred.groupBy('Survived', 'prediction').count().orderBy('Survived').show()